In [1]:
import pandas as pd
import pulp
from pulp import LpProblem, LpVariable, LpMaximize, lpSum, value, LpStatus
data = pd.read_csv('Dataset.csv', sep=';')
data = data.reset_index() 
preferences = pd.read_csv('Preferences.csv', sep=';')
print(data.columns)

Index(['index', 'Link', 'Price per month (total)',
       'Distance from CW (minutes walking)',
       'Distance from bus/tram stop (min walk)', 'Area of room (m^2)',
       'Parking price', 'Security deposit'],
      dtype='object')


# UTA

In [ ]:
criteria = [
    'Price per month (total)',
    'Distance from CW (minutes walking)',
    'Distance from bus/tram stop (min walk)',
    'Area of room (m^2)',
    'Parking price',
    'Security deposit',
]

maximize = {
    'Price per month (total)':                  False,
    'Distance from CW (minutes walking)':       False,
    'Distance from bus/tram stop (min walk)':   False,
    'Area of room (m^2)':                       True,
    'Parking price':                            False,
    'Security deposit':                         False,
}

norm = data[criteria].copy()
for c in criteria:
    lo, hi = data[c].min(), data[c].max()
    if hi == lo:
        norm[c] = 0.0
    elif maximize[c]:
        norm[c] = (data[c] - lo) / (hi - lo)
    else:
        norm[c] = (hi - data[c]) / (hi - lo)

data['Link'] = data['Link'].str.strip()
preferences['Link1'] = preferences['Link1'].str.strip()
preferences['Link2'] = preferences['Link2'].str.strip()
preferences['Relation'] = preferences['Relation'].str.strip()

url_to_idx = dict(zip(data['Link'], data.index))

pref_pairs = []
skipped = 0
for _, row in preferences.iterrows():
    l1, l2, rel = row['Link1'], row['Link2'], row['Relation']
    if l1 not in url_to_idx or l2 not in url_to_idx:
        print(f"  [WARN] URL not found in dataset, skipping: {l1[:60]}...")
        skipped += 1
        continue
    i1, i2 = url_to_idx[l1], url_to_idx[l2]
    if rel == '<':
        pref_pairs.append((i2, i1))
    elif rel == '>':
        pref_pairs.append((i1, i2))
    else:
        print(f"  [WARN] Unknown relation '{rel}', skipping.")
        skipped += 1

print(f"Loaded {len(pref_pairs)} preference pairs ({skipped} skipped).\n")

n_alts = len(data)
n_pref = len(pref_pairs)

# ── 4. Build LP ───────────────────────────────────────────────────────────────
prob = pulp.LpProblem("UTA_Model", pulp.LpMinimize)

# One marginal weight per criterion (sum to 1)
w = pulp.LpVariable.dicts("w", criteria, lowBound=0.07, upBound=0.5)
prob += pulp.lpSum(w[c] for c in criteria) == 1

# Overall utility for each alternative = weighted sum of normalised scores
U = [pulp.lpSum(w[c] * norm.loc[i, c] for c in criteria) for i in range(n_alts)]

# Slack variables for soft preference constraints
eps = pulp.LpVariable.dicts("eps", range(n_pref), lowBound=0)

# Objective: minimise total violation
prob += pulp.lpSum(eps[i] for i in range(n_pref))

# Preference constraints: U(better) >= U(worse) + delta - slack
DELTA = 0.01
for i, (better, worse) in enumerate(pref_pairs):
    prob += U[better] >= U[worse] + DELTA - eps[i]

prob.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"Status : {pulp.LpStatus[prob.status]}")
print(f"Total violation: {pulp.value(prob.objective):.6f}\n")

print("── Criterion weights ──")
for c in criteria:
    print(f"  {c:<45} {pulp.value(w[c]):.4f}")

print("\n── Alternative rankings ──")
scores = [(i, pulp.value(U[i])) for i in range(n_alts)]
scores.sort(key=lambda x: x[1], reverse=True)

for rank, (i, score) in enumerate(scores, 1):
    link = data.loc[i, 'Link']
    print(f"  #{rank:>2}  score={score:.4f}  idx={i}  {link}")

Loaded 12 preference pairs (0 skipped).

Status : Optimal
Total violation: 0.064865

── Criterion weights ──
  Price per month (total)                       0.3005
  Distance from CW (minutes walking)            0.1058
  Distance from bus/tram stop (min walk)        0.3837
  Area of room (m^2)                            0.0700
  Parking price                                 0.0700
  Security deposit                              0.0700

── Alternative rankings ──
  # 1  score=0.7951  idx=13  https://www.olx.pl/d/oferta/wynajem-pokoju-od-zaraz-stare-zegrze-29-CID3-ID19Wyt2.html?search_reason=search%7Corganic
  # 2  score=0.7653  idx=7  https://www.olx.pl/d/oferta/wynajem-pokoju-na-ratajach-CID3-ID19Sp6i.html?search_reason=search%7Cpromoted
  # 3  score=0.7151  idx=3  https://www.olx.pl/d/oferta/pokoj-do-wynajecia-CID3-ID19THhu.html?search_reason=search%7Corganic
  # 4  score=0.6928  idx=2  https://www.olx.pl/d/oferta/wynajme-maly-pokoj-os-armii-krajowej-CID3-ID19JlmW.html?search_reason=s